In [1]:
import torch
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from utils.evaluation import *

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
df_expert_45 = pd.read_csv("lemmatized/expert_core_45_lemmatized.csv")
df_expert_90 = pd.read_csv("lemmatized/expert_core_90_lemmatized.csv")
df_expert_180 = pd.read_csv("lemmatized/expert_core_180_lemmatized.csv")
df_test = pd.read_csv("lemmatized/test_lemmatized.csv")

In [4]:
def prepare_dataset(df, text_col='text_lemmatized', label_col='label'):
    df[text_col] = df[text_col].astype(str)

    mask = (df[text_col] != 'nan') & (df[text_col].str.strip() != '')
    df = df[mask].copy()

    dataset = Dataset.from_pandas(df[[text_col, label_col]])
    dataset = dataset.rename_column(text_col, "text")
    dataset = dataset.rename_column(label_col, "labels")
    return dataset

In [5]:
dataset_test = prepare_dataset(df_test)

dataset_expert_45 = prepare_dataset(df_expert_45)
dataset_expert_90 = prepare_dataset(df_expert_90)
dataset_expert_180 = prepare_dataset(df_expert_180)

In [6]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
def tokenize_function(examples):
    texts = examples["text"]
    
    cleaned_texts = []
    for t in texts:
        if t is None or (isinstance(t, float) and t != t):
            cleaned_texts.append("")
        elif not isinstance(t, str):
            cleaned_texts.append(str(t))
        else:
            cleaned_texts.append(t)
    
    return tokenizer(
        cleaned_texts, 
        truncation=True, 
        padding="max_length", 
        max_length=128
    )

In [8]:
tokenized_test = dataset_test.map(tokenize_function, batched=True)

tokenized_expert_45 = dataset_expert_45.map(tokenize_function, batched=True)
tokenized_expert_90 = dataset_expert_90.map(tokenize_function, batched=True)
tokenized_expert_180 = dataset_expert_180.map(tokenize_function, batched=True)

Map: 100%|██████████| 180/180 [00:00<00:00, 11313.02 examples/s]


In [9]:
columns_to_keep = ['input_ids', 'attention_mask', 'labels']

tokenized_test = tokenized_test.remove_columns(['text'])

tokenized_expert_45 = tokenized_expert_45.remove_columns(['text'])
tokenized_expert_90 = tokenized_expert_90.remove_columns(['text'])
tokenized_expert_180 = tokenized_expert_180.remove_columns(['text'])

In [10]:
def train_lora_model(train_ds, name, epochs, batch_size, lr, lora_r=8, lora_alpha=16):
    print(f"\n{'='*50}")
    print(f"Training LoRA: {name}")
    print(f"{'='*50}")
    
    # Базовая модель
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, 
        num_labels=3, 
        problem_type="single_label_classification"
    )
    
    # Конфигурация LoRA
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=lora_r,               # Ранг матрицы
        lora_alpha=lora_alpha,  # Коэффициент масштабирования
        lora_dropout=0.1,
        target_modules=["query", "value"] # Модули для адаптации в BERT
    )
    
    # Применение LoRA
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    
    # Аргументы обучения
    args = TrainingArguments(
        output_dir=f"./results_lora_{name}",
        logging_steps=50,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=0.01,
        report_to="none",
        fp16=True if device == "cuda" else False,
        seed=42,
        disable_tqdm=False,
        save_strategy="no" # Для экономии места, так как валидации нет
    )
    
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    return trainer

In [11]:
trainer_lora_45 = train_lora_model(
    tokenized_expert_45, "45", 
    epochs=50, batch_size=5, lr=5e-3, lora_r=16, lora_alpha=16
)


Training LoRA: 45


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1870.18it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

trainable params: 60,843 || all params: 29,255,550 || trainable%: 0.2080


Step,Training Loss
50,0.436648
100,0.002793
150,0.000638
200,0.000391
250,0.000297
300,0.000245
350,0.000243
400,0.000216
450,0.000199


In [12]:
trainer_lora_90 = train_lora_model(
    tokenized_expert_90, "90", 
    epochs=50, batch_size=10, lr=1e-3, lora_r=16, lora_alpha=16
)


Training LoRA: 90


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1140.27it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

trainable params: 60,843 || all params: 29,255,550 || trainable%: 0.2080


Step,Training Loss
50,0.834467
100,0.202396
150,0.023455
200,0.007195
250,0.004520
300,0.003469
350,0.002745
400,0.002688
450,0.002193


In [13]:
trainer_lora_180 = train_lora_model(
    tokenized_expert_180, "180", 
    epochs=38, batch_size=15, lr=7e-4, lora_r=16, lora_alpha=16
)


Training LoRA: 180


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1552.90it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

trainable params: 60,843 || all params: 29,255,550 || trainable%: 0.2080


Step,Training Loss
50,0.947009
100,0.443703
150,0.163913
200,0.067917
250,0.030272
300,0.017522
350,0.015517
400,0.011814
450,0.010280


In [14]:
res_lora_45 = evaluate_trainer(trainer_lora_45, tokenized_test, "LoRA (45 examples)")


Результаты модели: LoRA (45 examples)
   Accuracy:     0.5427
   F1 Macro:     0.5496
   F1 Weighted:  0.5496


In [15]:
res_lora_90 = evaluate_trainer(trainer_lora_90, tokenized_test, "LoRA (90 examples)")


Результаты модели: LoRA (90 examples)
   Accuracy:     0.5433
   F1 Macro:     0.5463
   F1 Weighted:  0.5463


In [16]:
res_lora_180 = evaluate_trainer(trainer_lora_180, tokenized_test, "LoRA (180 examples)")


Результаты модели: LoRA (180 examples)
   Accuracy:     0.5794
   F1 Macro:     0.5823
   F1 Weighted:  0.5823
